## Setup

In [10]:
import os
import re
import pandas as pd
import torch
from tqdm import tqdm
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor
from qwen_vl_utils import process_vision_info

In [11]:
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"

IMAGE_DIR = "./math/images/images/"
train = pd.read_csv("./math/train.csv")
test = pd.read_csv("./math/test.csv")

BATCH_SIZE = 4

In [12]:
MODEL_LOCAL_PATH = "/home/ub089/.cache/huggingface/hub/models--Qwen--Qwen2.5-VL-7B-Instruct/snapshots/cc594898137f460bfe9f0759e9844b3ce807cfb5"

In [13]:
train

,id,image_path,answer
0,0,images/0.jpg,20 ตารางเซนติเมตร
1,102,images/102.jpg,76
2,104,images/104.jpg,45
3,105,images/105.jpg,192
4,106,images/106.jpg,2
...,...,...,...
275,87,images/87.jpg,3749 จำนวน
276,91,images/91.jpg,125
277,95,images/95.jpg,2n+2
278,97,images/97.jpg,12


In [14]:
test

,id,image_path
0,1,images/1.jpg
1,10,images/10.jpg
2,100,images/100.jpg
3,101,images/101.jpg
4,103,images/103.jpg
...,...,...
415,92,images/92.jpg
416,93,images/93.jpg
417,94,images/94.jpg
418,96,images/96.jpg


In [15]:
test['answer'] = 0
test

,id,image_path,answer
0,1,images/1.jpg,0
1,10,images/10.jpg,0
2,100,images/100.jpg,0
3,101,images/101.jpg,0
4,103,images/103.jpg,0
...,...,...,...
415,92,images/92.jpg,0
416,93,images/93.jpg,0
417,94,images/94.jpg,0
418,96,images/96.jpg,0


## Normalize

In [16]:
def normalize_answer(text):
	if pd.isna(text): 
		return ""

	# 1. Lowercase and strip whitespace
	text = str(text).lower().strip()

	# 2. Thai digits to Arabic digits
	thai_digits = "๐๑๒๓๔๕๖๗๘๙"
	arabic_digits = "0123456789"
	trans = str.maketrans(thai_digits, arabic_digits)
	text = text.translate(trans)

	# 3. Strip math delimiters
	text = text.replace('$', '')

	# 4. Strip recognized units
	units = [
		'ตารางเซนติเมตร', 'ลูกบาศก์หน่วย', 'เซนติเมตร', 'องศา', 'หน่วย', 
		'จำนวน', 'วิธี', 'แบบ', 'ค่า', 'ร้อยละ', 'ดอลลาร์', 'บาท', 
		'degrees', 'square centimeters', 'years old'
	]
	for u in units:
		text = text.replace(u, '')

	# 5. Expand LaTeX
	text = re.sub(r'\\frac{([^{}]+)}{([^{}]+)}', r'(\1)/(\2)', text) # \frac{a}{b} -> (a)/(b)
	text = text.replace(r'\sqrt', 'sqrt')
	text = text.replace(r'\pi', 'pi')
	text = text.replace(r'\times', '*').replace(r'\cdot', '*')
	text = text.replace(r'\div', '/')
	text = text.replace(r'\pm', '+-')

	# Strip vector/line overheads (the braces are removed in the next step)
	text = text.replace(r'\overrightarrow', '').replace(r'\overline', '').replace(r'\vec', '')
	
	# Remove specific LaTeX formatting tokens
	for token in [r'\left', r'\right', r'\,', r'\;', r'\:', r'\!']:
		text = text.replace(token, '')

	# 6. Drop structural characters (whitespace and { } \ ,)
	text = re.sub(r'[\s{}\\,]', '', text)

	# 7. Drop redundant parentheses around pure integers: (3) -> 3
	text = re.sub(r'^\((\d+)\)$', r'\1', text)

	# 8. Integer canonicalization (e.g., 2.0 -> 2)
	try:
		val = float(text)
		if val.is_integer():
			text = str(int(val))
		else:
			text = str(val) # Handles standard decimals
	except ValueError:
		pass # If it's not a pure number (e.g., contains algebraic variables), leave it alone

	return text

## Inference

In [17]:
print(f"Loading Qwen2.5-VL-7B-Instruct directly from:\n{MODEL_LOCAL_PATH}")
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
	MODEL_LOCAL_PATH,
	torch_dtype=torch.bfloat16,
	device_map="cuda:0",
	local_files_only=True,
)

Loading Qwen2.5-VL-7B-Instruct directly from:
/home/ub089/.cache/huggingface/hub/models--Qwen--Qwen2.5-VL-7B-Instruct/snapshots/cc594898137f460bfe9f0759e9844b3ce807cfb5


Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

In [18]:
processor = AutoProcessor.from_pretrained(
    MODEL_LOCAL_PATH,
    local_files_only=True
)

processor.tokenizer.padding_side = 'left'

The image processor of type `Qwen2VLImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. Note that this behavior will be extended to all models in a future release.


In [19]:
system_prompt = (
	"You are an expert math solver. Look at the provided image and solve the mathematical problem step-by-step. "
	"Write out your full reasoning. "
	"When you are finished, you MUST output the final answer on a new line starting exactly with 'FINAL_ANSWER: ' "
	"followed by the value."
)

In [20]:
train['raw_prediction'] = "Error"
train['norm_prediction'] = "error"
train['norm_truth'] = train['answer'].apply(normalize_answer)

for i in tqdm(range(0, len(train), BATCH_SIZE)):
	batch_df = train.iloc[i : i + BATCH_SIZE]

	batch_messages = []
	valid_indices = [] 

	for idx, row in batch_df.iterrows():
		img_path = f"{IMAGE_DIR}{row['id']}.jpg"
		if os.path.exists(img_path):
			valid_indices.append(idx)
			batch_messages.append([
				{
					"role": "user",
					"content": [
						{"type": "image", "image": img_path},
						{"type": "text", "text": system_prompt}
					]
				}
			])

	if not batch_messages:
		continue 

	texts = [processor.apply_chat_template(msg, tokenize=False, add_generation_prompt=True) for msg in batch_messages]
	image_inputs, video_inputs = process_vision_info(batch_messages)

	inputs = processor(
		text=texts, images=image_inputs, videos=video_inputs, padding=True, return_tensors="pt"
	).to(model.device)

	with torch.no_grad():
		generated_ids = model.generate(**inputs, max_new_tokens=1024, do_sample=False)

	generated_ids_trimmed = [out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)]
	output_texts = processor.batch_decode(generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False)

	for out_text, df_idx in zip(output_texts, valid_indices):
		if "FINAL_ANSWER:" in out_text:
			raw_ans = out_text.split("FINAL_ANSWER:")[-1].strip()
		else:
			raw_ans = out_text.split('\n')[-1].strip()

		# Insert directly into the dataframe at the correct index
		train.at[df_idx, 'raw_prediction'] = raw_ans
		train.at[df_idx, 'norm_prediction'] = normalize_answer(raw_ans)

 89%|████████▊ | 62/70 [34:49<04:29, 33.70s/it]


KeyboardInterrupt: 

## Evaluate

In [ ]:
train['is_correct'] = train['norm_prediction'] == train['norm_truth']
accuracy = train['is_correct'].mean()

print(f"\n======================================")
print(f"LOCAL VALIDATION ACCURACY: {accuracy * 100:.2f}%")
print(f"======================================\n")

NameError: name 'predictions' is not defined

## Save to csv

In [ ]:
print(f"Starting BATCHED Chain-of-Thought inference on {len(test)} TEST images...")

# Pre-allocate array with safe defaults
predicted_answers = ["0"] * len(test)

for i in tqdm(range(0, len(test), BATCH_SIZE)):
	batch_df = test.iloc[i : i + BATCH_SIZE]

	batch_messages = []
	valid_indices = [] 

	for idx, row in batch_df.iterrows():
		img_path = f"{IMAGE_DIR}{row['id']}.jpg"
		if os.path.exists(img_path):
			valid_indices.append(idx)
			batch_messages.append([
				{
					"role": "user",
					"content": [
						{"type": "image", "image": img_path},
						{"type": "text", "text": system_prompt}
					]
				}
			])

	if not batch_messages:
		continue 
		
	texts = [processor.apply_chat_template(msg, tokenize=False, add_generation_prompt=True) for msg in batch_messages]
	image_inputs, video_inputs = process_vision_info(batch_messages)

	inputs = processor(
		text=texts, images=image_inputs, videos=video_inputs, padding=True, return_tensors="pt"
	).to(model.device)

	with torch.no_grad():
		generated_ids = model.generate(**inputs, max_new_tokens=1024, do_sample=False) # Removed temperature=0.0

	generated_ids_trimmed = [out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)]
	output_texts = processor.batch_decode(generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False)

	for out_text, df_idx in zip(output_texts, valid_indices):
		if "FINAL_ANSWER:" in out_text:
			raw_ans = out_text.split("FINAL_ANSWER:")[-1].strip()
		else:
			raw_ans = out_text.split('\n')[-1].strip()

		clean_ans = normalize_answer(raw_ans)
		if clean_ans == "":
			clean_ans = "0"
			
		predicted_answers[df_idx] = clean_ans

In [ ]:
predicted_answers = []

print(f"Starting Chain-of-Thought inference on {len(test)} TEST images...")

for index, row in tqdm(test.iterrows(), total=len(test)):
	img_path = f"{IMAGE_DIR}{row['id']}.jpg"
	
	if not os.path.exists(img_path):
		predicted_answers.append("0")
		continue

	messages = [
		{
			"role": "user",
			"content": [
				{"type": "image", "image": img_path},
				{"type": "text", "text": system_prompt}
			]
		}
	]

	text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
	image_inputs, video_inputs = process_vision_info(messages)

	inputs = processor(
		text=[text], images=image_inputs, videos=video_inputs, padding=True, return_tensors="pt"
	).to(model.device)

	with torch.no_grad():
		generated_ids = model.generate(
			**inputs, 
			max_new_tokens=1024, 
			temperature=0.0,
			do_sample=False
		)

	generated_ids_trimmed = [
		out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
	]

	output_text = processor.batch_decode(
		generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
	)[0]

	if "FINAL_ANSWER:" in output_text:
		raw_final_answer = output_text.split("FINAL_ANSWER:")[-1].strip()
	else:
		raw_final_answer = output_text.split('\n')[-1].strip()

	final_clean_answer = normalize_answer(raw_final_answer)

	if final_clean_answer == "":
		final_clean_answer = "0"

	predicted_answers.append(final_clean_answer)

In [ ]:
test['answer'] = predicted_answers
submission = test.copy().drop(columns=['image_path'])
submission.to_csv("submission.csv", index=False)
print("\nSuccessfully generated submission.csv in record time!")